In [1]:

# ============================================================
# TASK 5: CONTENT-BASED TED TALKS RECOMMENDATION ENGINE
# Notebook: 05_content_based_recommendation.ipynb
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

DATA_PATH = "/content/ted_talks_cleaned.csv"

if not Path(DATA_PATH).exists():
    from google.colab import files
    print("Upload ted_talks_cleaned.csv")
    files.upload()

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = ["talk_id", "title", "combined_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# ------------------------------------------------------------
# 3. PREPARE TEXT DATA
# ------------------------------------------------------------

df["title"] = df["title"].fillna("").astype(str)

df["combined_text"] = (
    df["combined_text"]
    .fillna("")
    .astype(str)
)

df = df[
    (df["title"].str.strip() != "") &
    (df["combined_text"].str.strip() != "")
].copy()

df.reset_index(drop=True, inplace=True)

print("Usable TED Talks:", len(df))

# ------------------------------------------------------------
# 4. TF-IDF VECTORIZATION (EXECUTED ONLY ONCE)
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["combined_text"]
)

print("\nTF-IDF Matrix Shape:", tfidf_matrix.shape)

# ------------------------------------------------------------
# 5. COSINE SIMILARITY (EXECUTED ONLY ONCE)
# ------------------------------------------------------------

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

# ------------------------------------------------------------
# 6. SEARCH FOR TED TALKS BY TITLE
# ------------------------------------------------------------

def search_talks(keyword, limit=10):

    matches = df[
        df["title"].str.contains(
            keyword,
            case=False,
            na=False,
            regex=False
        )
    ]

    return matches[
        ["talk_id", "title"]
    ].head(limit).reset_index(drop=True)


# ------------------------------------------------------------
# 7. REUSABLE RECOMMENDATION FUNCTION
# ------------------------------------------------------------

def recommend_talks(title, top_n=5):

    if not isinstance(title, str) or not title.strip():
        print("Please enter a valid TED Talk title.")
        return pd.DataFrame()

    if top_n <= 0:
        print("top_n must be greater than zero.")
        return pd.DataFrame()

    # Find exact matching title
    matches = df[
        df["title"].str.strip().str.lower()
        == title.strip().lower()
    ]

    if matches.empty:
        print("TED Talk not found:", title)
        print("Use search_talks() to find available titles.")
        return pd.DataFrame()

    if len(matches) > 1:
        print("Multiple talks have this title. Using the first match.")

    selected_index = matches.index[0]

    # Get similarity scores from precomputed matrix
    scores = similarity_matrix[selected_index].copy()

    # Exclude selected talk
    scores[selected_index] = -1

    # Sort in descending order
    recommended_indices = np.argsort(scores)[::-1]

    recommended_indices = [
        idx for idx in recommended_indices
        if scores[idx] >= 0
    ][:top_n]

    # Prepare results
    recommendations = df.iloc[
        recommended_indices
    ][["talk_id", "title"]].copy()

    recommendations["similarity_score"] = [
        round(float(scores[idx]), 4)
        for idx in recommended_indices
    ]

    recommendations = recommendations.reset_index(drop=True)

    return recommendations


# ------------------------------------------------------------
# 8. TEST CASE 1: SEARCH FOR A TALK
# ------------------------------------------------------------

print("\nAVAILABLE TED TALKS:")
display(search_talks("climate", limit=10))

# ------------------------------------------------------------
# 9. TEST CASE 2: SELECT THREE ACTUAL TALKS
# ------------------------------------------------------------

selected_talks = (
    df["title"]
    .drop_duplicates()
    .sample(
        n=min(3, df["title"].nunique()),
        random_state=42
    )
    .tolist()
)

print("\nRECOMMENDATION TESTING")

for number, title in enumerate(selected_talks, start=1):

    print("\n" + "=" * 70)
    print(f"TEST CASE {number}")
    print("Selected TED Talk:", title)
    print("=" * 70)

    result = recommend_talks(title, top_n=5)

    display(result)

# ------------------------------------------------------------
# 10. TEST CASE 3: INVALID TITLE
# ------------------------------------------------------------

print("\nINVALID TITLE TEST:")

invalid_result = recommend_talks(
    "This TED Talk Does Not Exist",
    top_n=5
)

# ------------------------------------------------------------
# 11. SAVE RECOMMENDATION RESULTS
# ------------------------------------------------------------

all_results = []

for title in selected_talks:

    result = recommend_talks(title, top_n=5)

    if not result.empty:

        result.insert(0, "selected_talk", title)

        all_results.append(result)

if all_results:

    results_df = pd.concat(
        all_results,
        ignore_index=True
    )

    results_df.to_csv(
        "/content/task5_recommendation_results.csv",
        index=False
    )

    print("\nRecommendation results saved successfully.")

    display(results_df)

# ------------------------------------------------------------
# 12. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TASK 5 COMPLETION REPORT")
print("=" * 60)

print("Total TED Talks:", len(df))
print("TF-IDF Features:", tfidf_matrix.shape[1])
print("TF-IDF Matrix:", tfidf_matrix.shape)
print("Similarity Matrix:", similarity_matrix.shape)

print("\nFunctions implemented:")
print("1. search_talks(keyword)")
print("2. recommend_talks(title, top_n=5)")

print("\nObservations:")
print("- Recommendations are generated using textual similarity.")
print("- The selected talk is excluded from its own results.")
print("- The similarity matrix is calculated only once.")
print("- Invalid titles are handled appropriately.")
print("- Different selected talks can produce different recommendations.")

print("\nTASK 5 COMPLETED SUCCESSFULLY!")


Dataset loaded successfully!
Dataset shape: (4005, 24)
Usable TED Talks: 4005

TF-IDF Matrix Shape: (4005, 10000)
Similarity Matrix Shape: (4005, 4005)

AVAILABLE TED TALKS:


,talk_id,title
0,1,Averting the climate crisis
1,62,Global priorities bigger than climate change
2,192,A critical look at geoengineering against clim...
3,243,New thinking on the climate crisis
4,682,The science behind a climate headline
5,954,Discovering ancient climates in oceans and ice
6,1380,Why I must speak out about climate change
7,1583,Let's prepare for our new climate
8,1683,How to fight desertification and reverse clima...
9,1988,The emergent patterns of climate change



RECOMMENDATION TESTING

TEST CASE 1
Selected TED Talk: Glorious visions in animation and performance


,talk_id,title,similarity_score
0,25874,The princess who rewrote history,0.1397
1,2273,An 11-year-old prodigy performs old-school jazz,0.1220
2,41656,The Opposites Game,0.1101
3,729,An all-star set,0.1094
4,241,A new kind of music video,0.1042



TEST CASE 2
Selected TED Talk: Medicine for the 99 percent


,talk_id,title,similarity_score
0,1627,"Pool medical patents, save lives",0.2893
1,56527,Why are drug prices so high? Investigating the...,0.2367
2,767,Innovating to zero!,0.2144
3,61303,The quest for the coronavirus vaccine,0.2114
4,85,My wish: Rebuilding Rwanda,0.2084



TEST CASE 3
Selected TED Talk: What is so special about the human brain?


,talk_id,title,similarity_score
0,24026,What percentage of your brain do you use?,0.3246
1,984,The brain in your gut,0.2565
2,1308,The quest to understand consciousness,0.1997
3,1674,How a fly flies,0.1979
4,724,The neurons that shaped civilization,0.1920



INVALID TITLE TEST:
TED Talk not found: This TED Talk Does Not Exist
Use search_talks() to find available titles.

Recommendation results saved successfully.


,selected_talk,talk_id,title,similarity_score
0,Glorious visions in animation and performance,25874,The princess who rewrote history,0.1397
1,Glorious visions in animation and performance,2273,An 11-year-old prodigy performs old-school jazz,0.1220
2,Glorious visions in animation and performance,41656,The Opposites Game,0.1101
3,Glorious visions in animation and performance,729,An all-star set,0.1094
4,Glorious visions in animation and performance,241,A new kind of music video,0.1042
5,Medicine for the 99 percent,1627,"Pool medical patents, save lives",0.2893
6,Medicine for the 99 percent,56527,Why are drug prices so high? Investigating the...,0.2367
7,Medicine for the 99 percent,767,Innovating to zero!,0.2144
8,Medicine for the 99 percent,61303,The quest for the coronavirus vaccine,0.2114
9,Medicine for the 99 percent,85,My wish: Rebuilding Rwanda,0.2084



TASK 5 COMPLETION REPORT
Total TED Talks: 4005
TF-IDF Features: 10000
TF-IDF Matrix: (4005, 10000)
Similarity Matrix: (4005, 4005)

Functions implemented:
1. search_talks(keyword)
2. recommend_talks(title, top_n=5)

Observations:
- Recommendations are generated using textual similarity.
- The selected talk is excluded from its own results.
- The similarity matrix is calculated only once.
- Invalid titles are handled appropriately.
- Different selected talks can produce different recommendations.

TASK 5 COMPLETED SUCCESSFULLY!
